### bulkformer-latest feature extraction

This notebook uses **Stage1 epoch 17 / update 126,055** (101.3M parameters, 19,973 genes, 768 features). It follows the existing BulkFormer notebook: prepare data, align genes, then extract sample, gene, and expression features.

See the [model introduction](../docs/models/bulkformer-latest.md) for architecture and evaluation results. The setup cell locates the repository root automatically. See the [147M comparison](../benchmarks/phenotype/README.md) for the matched phenotype benchmark.


#### 1. Environment and pretrained weights

```bash
conda env create -f configs/bulkformer-latest.yaml
conda activate bulkformer-latest
jupyter lab
```

Download the complete [model.pt](https://drive.google.com/file/d/1ly4CNE6XHqHKSP1a5c0UGUeaSVC0Was1/view?usp=drivesdk) and the five resource files from the [Google Drive model folder](https://drive.google.com/drive/folders/1AfekiOBNs6GzPBbA9BDlpuoolZIWbGQ2) into `model/bulkformer-latest/`, preserving their names:

```text
model/bulkformer-latest/
  model.pt
  model_config.json
  gene_vocab.csv
  edge_index.pt
  edge_weight.pt
  manifest.json
```

If Drive wraps the folder download in a ZIP, extract it first. The preparation cell verifies each file against the published SHA256 before loading the model.

Select your GPU **before importing torch**. If CUDA is already initialized, restart the notebook kernel after changing the GPU selection.


In [ ]:
import os

# Choose a physical GPU here before importing torch.
# setdefault also respects a GPU selected when launching Jupyter.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "utils/latest").is_dir()
     and (candidate / "model/downloads/bulkformer-latest.json").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook inside the cloned BulkFormer repository")
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import torch

from utils.latest import (
    prepare_latest_bundle, load_latest_model,
    normalize_data, main_gene_selection, extract_feature,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = ("bf16" if torch.cuda.is_bf16_supported() else "fp16") if device.type == "cuda" else "fp32"
BATCH_SIZE = 1
print("Device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")


#### 2. Load the released model

The bundle includes the exact graph and ordered vocabulary required by this checkpoint. `load_latest_model` verifies the artifacts and strictly loads the full encoder and expression head.


In [ ]:
MODEL_DIR = prepare_latest_bundle(REPO_ROOT / "model/bulkformer-latest", REPO_ROOT / "model")

model, gene_vocab, manifest = load_latest_model(MODEL_DIR, device=device)
gene_list = gene_vocab["canonical_ensg"].tolist()
print(manifest["release_name"])
print("Genes:", len(gene_list), "Feature dimension:", model.config.dim)
print("Checkpoint:", manifest["checkpoint_metadata"])


#### 3. Read raw counts or normalized expression

The included CSV contains **four synthetic samples**, observed zeros, and eight omitted genes. Replace `INPUT_PATH` with your own CSV (samples × Ensembl genes). Ensembl version suffixes are removed; duplicate resulting IDs must be resolved upstream.

For raw counts, TPM normalization uses vocabulary-matched genes and the bundled gene lengths. For normalized input, choose the matching `INPUT_KIND`; the model requires **natural `ln(TPM+1)`**. Gene symbols are annotation only. Missing entries may be NaN; measured zero must remain zero.


In [ ]:
INPUT_PATH = REPO_ROOT / "data/latest_demo_counts.csv"
INPUT_KIND = "raw_count"  # alternatives: tpm, ln_tpm1, log2_tpm1

input_df = pd.read_csv(INPUT_PATH, index_col=0)
if INPUT_KIND == "raw_count":
    expression_df = normalize_data(input_df, gene_vocab)
elif INPUT_KIND == "tpm":
    if (input_df < 0).any().any():
        raise ValueError("TPM must be non-negative")
    expression_df = np.log1p(input_df)
elif INPUT_KIND == "ln_tpm1":
    expression_df = input_df.copy()
elif INPUT_KIND == "log2_tpm1":
    expression_df = input_df * np.log(2.0)
else:
    raise ValueError("Choose a supported INPUT_KIND")
print("Input samples × genes:", input_df.shape)


#### 4. Align to the full gene vocabulary

Unavailable genes are filled with `-10`; measured zeros remain `0`. All 19,973 gene tokens are processed. Sample pooling excludes missing genes separately for each sample and includes observed zeros. No additional random masking is used for feature extraction.


In [ ]:
aligned_df, missing_genes, gene_metadata = main_gene_selection(expression_df, gene_list)
X = aligned_df.to_numpy(dtype=np.float32)
sample_ids = aligned_df.index.to_numpy()
print("Aligned shape:", X.shape)
print("Globally unavailable genes:", len(missing_genes))
print("Observed gene coverage:", (X != -10).mean(axis=1))
print("Measured zero counts:", (X == 0).sum(axis=1))


#### 5. Sample-level features

Observed-gene mean pooling produces one 768-dimensional vector per sample, matching the pooling used for the reported phenotype benchmark. The notebook uses batch size 1; increase it if your GPU has enough memory.


In [ ]:
sample_features = extract_feature(
    model, X, output_feature_type="sample_level", aggregate_type="mean",
    batch_size=BATCH_SIZE, amp_dtype=AMP_DTYPE,
)
sample_feature_df = pd.DataFrame(sample_features, index=sample_ids)
print("Sample feature shape:", sample_features.shape)
sample_feature_df.iloc[:, :5]


#### 6. Contextual gene-level features

Extract a small subset of output genes to keep returned arrays compact. The encoder still processes the complete transcriptome. Each gene vector depends on its sample context. The static trainable gene identity table is a different representation.


In [ ]:
selected_gene_indices = np.arange(8, dtype=np.int64)
selected_genes = gene_vocab.iloc[selected_gene_indices][["canonical_ensg", "gene_symbol"]]
gene_features = extract_feature(
    model, X, output_feature_type="gene_level",
    interested_gene_idx=selected_gene_indices,
    batch_size=BATCH_SIZE, amp_dtype=AMP_DTYPE,
)
print("Selected gene feature shape:", gene_features.shape)
selected_genes


#### 7. Expression predictions

The shared head returns a prediction for every gene on the learned `ln(TPM+1)` target scale. Its output is unconstrained, so negative predictions are possible. These are model predictions rather than measured expression values. For imputation of selected observed genes, replace those input values with `-10` before inference and retain the true values separately.


In [ ]:
expression_predictions = extract_feature(
    model, X, output_feature_type="expr_level",
    batch_size=BATCH_SIZE, amp_dtype=AMP_DTYPE,
)
prediction_df = pd.DataFrame(expression_predictions, index=sample_ids, columns=gene_list)
print("Expression prediction shape:", prediction_df.shape)
prediction_df.iloc[:, :5]


#### 8. Save features

The saved sample order and selected gene IDs allow downstream results to be aligned with their inputs. The example output directory can be changed below.


In [ ]:
OUTPUT_DIR = REPO_ROOT / "outputs/latest_demo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sample_feature_df.to_csv(OUTPUT_DIR / "sample_features.csv")
prediction_df.to_csv(OUTPUT_DIR / "expression_predictions.csv")
np.savez_compressed(
    OUTPUT_DIR / "selected_gene_features.npz", features=gene_features,
    sample_ids=sample_ids.astype(str),
    gene_ids=selected_genes["canonical_ensg"].to_numpy(dtype=str),
)
print("Saved outputs to:", OUTPUT_DIR.resolve())
